# HITL A V3: GPU Draft and Confidence-Map Generator

**Runtime:** T4 GPU.

This generator saves both:

- a binary draft mask at the labelling threshold; and
- the full pixel confidence map as a NumPy array.

The CPU reviewer uses the confidence map to render graduated red intensity from confidence 0.20 to 1.00. Reviewed sources and current drafts are skipped automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject');SOURCE_DIR=ROOT/'source'/'annotated_only';MODEL_PATH=ROOT/'training_runs'/'20261006_150912_yolo11n-seg'/'weights'/'best.pt';HITL=ROOT/'human_in_the_middle';DRAFT_MASK_DIR=HITL/'draft_masks';CONFIDENCE_DIR=HITL/'confidence_maps';DRAFT_AUDIT_DIR=HITL/'draft_audit';REVIEW_AUDIT_DIR=HITL/'audit'
PREDICTION_CONFIDENCE=.20;TILE_SIZE=640;TILE_STRIDE=320;MODEL_IMAGE_SIZE=640;MINIMUM_DRAFT_COMPONENT_AREA=10;FORCE_REBUILD_ALL=False
for f in [SOURCE_DIR,DRAFT_MASK_DIR,CONFIDENCE_DIR,DRAFT_AUDIT_DIR,REVIEW_AUDIT_DIR]:f.mkdir(parents=True,exist_ok=True)

In [ ]:
import cv2,json,hashlib,torch
import numpy as np
from ultralytics import YOLO
from datetime import datetime,timezone
SUPPORTED={'.jpg','.jpeg','.png','.tif','.tiff','.bmp'}
if not MODEL_PATH.exists():raise RuntimeError(f'Model not found {MODEL_PATH}')
if not torch.cuda.is_available():raise RuntimeError('Select T4 GPU')
MODEL=YOLO(str(MODEL_PATH))
def files(f):return sorted(p for p in Path(f).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)
def md5(p):
 h=hashlib.md5()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def reviewed():
 out=set()
 for p in REVIEW_AUDIT_DIR.glob('Page*.json'):
  try:
   v=json.loads(p.read_text()).get('source_md5')
   if v:out.add(v)
  except:pass
 return out
def pos(n):
 if n<=TILE_SIZE:return [0]
 p=list(range(0,n-TILE_SIZE+1,TILE_STRIDE))
 if p[-1]!=n-TILE_SIZE:p.append(n-TILE_SIZE)
 return p
def confidence_map(gray):
 colour=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);h,w=gray.shape;ph,pw=max(h,TILE_SIZE),max(w,TILE_SIZE);canvas=np.full((ph,pw,3),255,np.uint8);canvas[:h,:w]=colour;conf=np.zeros((ph,pw),np.float32)
 for y in pos(ph):
  for x in pos(pw):
   r=MODEL.predict(canvas[y:y+TILE_SIZE,x:x+TILE_SIZE],conf=PREDICTION_CONFIDENCE,imgsz=MODEL_IMAGE_SIZE,device=0,verbose=False,retina_masks=True)[0]
   if r.masks is None:continue
   for mask,score in zip(r.masks.data.cpu().numpy(),r.boxes.conf.cpu().numpy()):
    mask=cv2.resize(mask,(TILE_SIZE,TILE_SIZE),interpolation=cv2.INTER_NEAREST)>.5;region=conf[y:y+TILE_SIZE,x:x+TILE_SIZE];region[mask&(score>region)]=score
 return conf[:h,:w]

done=reviewed();model_hash=md5(MODEL_PATH);generated=skipped_reviewed=skipped_current=0
for source in files(SOURCE_DIR):
 sh=md5(source)
 if sh in done:skipped_reviewed+=1;continue
 mask_path=DRAFT_MASK_DIR/f'{source.stem}_draft.png';conf_path=CONFIDENCE_DIR/f'{source.stem}_confidence.npy';audit_path=DRAFT_AUDIT_DIR/f'{source.stem}.json';old=json.loads(audit_path.read_text()) if audit_path.exists() else {}
 current=(not FORCE_REBUILD_ALL and mask_path.exists() and conf_path.exists() and old.get('source_md5')==sh and old.get('model_md5')==model_hash and old.get('prediction_confidence')==PREDICTION_CONFIDENCE)
 if current:skipped_current+=1;continue
 gray=cv2.imread(str(source),0)
 if gray is None:print('FAILED',source);continue
 conf=confidence_map(gray);binary=(conf>=PREDICTION_CONFIDENCE).astype(np.uint8)*255;np.save(conf_path,conf.astype(np.float16));cv2.imwrite(str(mask_path),binary);audit_path.write_text(json.dumps({'created_utc':datetime.now(timezone.utc).isoformat(),'source_file':str(source),'source_md5':sh,'model_path':str(MODEL_PATH),'model_md5':model_hash,'prediction_confidence':PREDICTION_CONFIDENCE,'draft_pixels':int(np.sum(binary>0)),'confidence_map':str(conf_path),'draft_mask':str(mask_path),'shape':list(gray.shape)},indent=2));generated+=1;print('Generated',source.name)
print('Generated',generated,'Skipped reviewed',skipped_reviewed,'Skipped current',skipped_current)